## **집계연산**

### **PySpark 설정**

In [11]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext

In [12]:
# [+] SparkConf, SparkContext 객체 생성하기
SparkContext.getOrCreate().stop()
conf = SparkConf().setMaster("local[*]").setAppName("Reduction Operations")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 08:43:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/08 08:43:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [13]:
sc = SparkContext(conf=conf)



26/10/08 08:43:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### **reduce( ) 메서드**

In [14]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x, y: x + y)

15

In [15]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()

[1, 2, 3, 4, 5]

In [16]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [17]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
rdd.reduce(lambda x, y: x + y)

15

In [18]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5],2).reduce(lambda x, y: (x * 2) + y)

33

In [19]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 4).reduce(lambda x,y: (x * 2) + y)

35

### **10000 개 정수 난수의 총합 계산하기**

In [20]:
%pip install pyspark --root-user-action=ignore


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [22]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0, 100, size = 10000)
x[:100]

array([38, 80, 43, 72, 53, 90, 26, 40, 86, 37, 55, 16, 35, 44, 95, 66, 64,
        2, 96, 49, 15, 21, 14, 68, 52, 27, 60,  6, 33, 39, 66, 62, 19, 65,
       66, 31, 50, 36, 88, 96, 19, 75, 32, 18, 70, 68, 75,  0, 20, 45, 58,
       88, 17, 38, 50, 78, 38, 42, 47, 42, 13, 17, 47, 17, 92, 37, 47, 51,
        7, 14, 44,  6, 92, 38, 86, 32, 89, 55,  7, 98,  4, 15, 32, 40, 76,
       36,  5, 17, 77, 26, 43, 93,  3, 68,  8, 79, 70, 68, 45, 25])

In [23]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x, 8)

In [24]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

Partition 0: [np.int64(38), np.int64(80), np.int64(43), np.int64(72), np.int64(53), np.int64(90), np.int64(26), np.int64(40), np.int64(86), np.int64(37), np.int64(55), np.int64(16), np.int64(35), np.int64(44), np.int64(95), np.int64(66), np.int64(64), np.int64(2), np.int64(96), np.int64(49), np.int64(15), np.int64(21), np.int64(14), np.int64(68), np.int64(52), np.int64(27), np.int64(60), np.int64(6), np.int64(33), np.int64(39), np.int64(66), np.int64(62), np.int64(19), np.int64(65), np.int64(66), np.int64(31), np.int64(50), np.int64(36), np.int64(88), np.int64(96), np.int64(19), np.int64(75), np.int64(32), np.int64(18), np.int64(70), np.int64(68), np.int64(75), np.int64(0), np.int64(20), np.int64(45), np.int64(58), np.int64(88), np.int64(17), np.int64(38), np.int64(50), np.int64(78), np.int64(38), np.int64(42), np.int64(47), np.int64(42), np.int64(13), np.int64(17), np.int64(47), np.int64(17), np.int64(92), np.int64(37), np.int64(47), np.int64(51), np.int64(7), np.int64(14), np.int64(4

In [25]:
# [+] x+y 집계 연산
rdd.reduce(lambda x, y: x + y)

np.int64(495742)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [26]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x, y: x + y)

19

In [27]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4], 2).fold(1, lambda x, y: x + y)

13

In [28]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x, y: x * y)

0

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [29]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([3,10,7,5,12,8,10]).groupBy(lambda x: x % 2).collect()

In [30]:
# [+] groupBy() 결과 출력하기
for k,v in res:
    print(k,list(v))

0 [10, 12, 8, 10]
1 [3, 7, 5]


### **aggregate( ) 메서드**

In [31]:
# int -> (합, 곱) 튜플로 만들기
seqOp = lambda acc, v: (acc[0] + v, acc[1] * v)
combOp = lambda a, b: (a[0] + b[0], a[1] * b[1])

sc.parallelize([1, 2, 3, 4]).map(lambda x: (x, x)).reduce(
    lambda a, b: (a[0] + b[0], a[1] * b[1])
)


(10, 24)

In [32]:
sc.parallelize([1, 2, 3, 4]).aggregate(
    (0, 0),
    lambda acc, v: (acc[0] + v, acc[1] + 1),   # 파티션 안: 튜플 + 숫자
    lambda a, b: (a[0] + b[0], a[1] + b[1])    # 파티션 간: 튜플 + 튜플
)
# (10, 4)

(10, 4)

In [33]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [34]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0, 0), seqOp, combOp)

(10, 4)